In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# QUALITATIVE RISK MODEL INTEGRATION
# CELL 1 — PROJECT INITIALIZATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\Users\jpiza\PharmaOpt")

OUTPUT_DIR = PROJECT_ROOT / "outputs"

RISK_FILE = OUTPUT_DIR / "recall_risk_signals_v02.csv"

print("=" * 70)
print("PHARMAOPT — QUALITATIVE RISK MODEL INTEGRATION")
print("=" * 70)

print("\nProject:", PROJECT_ROOT)
print("Risk dataset:", RISK_FILE)
print("File available:", RISK_FILE.exists())

if not RISK_FILE.exists():
    raise FileNotFoundError(
        f"Cannot locate qualitative risk dataset: {RISK_FILE}"
    )

print("\nInitialization successful.")


In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# CELL 2 — VALIDATED QUALITATIVE RISK DATA
# ============================================================

import pandas as pd

print("=" * 70)
print("PHARMAOPT — QUALITATIVE RISK DATA VALIDATION")
print("=" * 70)

# 1. Load data while preserving original identifier values
raw_risk_df = pd.read_csv(
    RISK_FILE,
    dtype={"recall_number": "string"},
    keep_default_na=False
)

# 2. Identify missing and placeholder recall numbers
recall_ids = (
    raw_risk_df["recall_number"]
    .astype("string")
    .str.strip()
)

invalid_values = {
    "", "N/A", "NA", "NULL", "NONE", "UNKNOWN"
}

invalid_mask = (
    recall_ids.isna()
    | recall_ids.str.upper().isin(invalid_values)
).fillna(True)

# 3. Preserve invalid records for review
invalid_records = raw_risk_df.loc[invalid_mask].copy()

review_file = OUTPUT_DIR / "recall_risk_invalid_ids_review.csv"

invalid_records.to_csv(review_file, index=False)

# 4. Create integration-ready dataset
risk_df = raw_risk_df.loc[~invalid_mask].copy()

risk_df["recall_number"] = (
    recall_ids.loc[~invalid_mask].astype(str)
)

# 5. Identify the 13 risk categories
risk_categories = [
    col for col in risk_df.columns
    if col.endswith("_risk")
]

# 6. Validate dataset integrity
assert len(raw_risk_df) == 17974
assert len(invalid_records) == 1
assert len(risk_df) == 17973
assert len(risk_categories) == 13
assert risk_df["recall_number"].is_unique

for col in risk_categories:
    assert risk_df[col].notna().all()
    assert risk_df[col].isin([0, 1]).all()

assert (
    risk_df[risk_categories].sum(axis=1)
    == risk_df["risk_signal_count_v2"]
).all()

assert risk_df["signal_source"].eq("rule_based").all()

# 7. Save the verified integration dataset
INTEGRATION_FILE = (
    OUTPUT_DIR / "recall_risk_signals_v02_integration.csv"
)

risk_df.to_csv(INTEGRATION_FILE, index=False)

# 8. Final summary
print("\nVALIDATION PASSED")
print("-" * 70)
print("Source records:", len(raw_risk_df))
print("Integration-ready records:", len(risk_df))
print("New invalid IDs identified:", len(invalid_records))
print("Risk categories:", len(risk_categories))
print("Duplicate IDs:", risk_df["recall_number"].duplicated().sum())
print("Integration file exists:", INTEGRATION_FILE.exists())

print("\nIntegration file:", INTEGRATION_FILE)
print("Review file:", review_file)

display(risk_df.head())


In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# CELL 3 — INVESTIGATE MISSING RECALL IDENTIFIER
# ============================================================

import pandas as pd

print("=" * 70)
print("PHARMAOPT — RECALL IDENTIFIER INVESTIGATION")
print("=" * 70)

# Load the CSV without automatic missing-value conversion
raw_risk_df = pd.read_csv(
    RISK_FILE,
    dtype={"recall_number": "string"},
    keep_default_na=False
)

# Compare with Pandas' default CSV loading
default_risk_df = pd.read_csv(
    RISK_FILE,
    dtype={"recall_number": "string"}
)

missing_mask = default_risk_df["recall_number"].isna()

print("\nTotal records:", len(raw_risk_df))

print("Missing IDs with default loading:",
      int(missing_mask.sum()))

print("Original values interpreted as missing:")

for value in raw_risk_df.loc[missing_mask, "recall_number"]:
    print(repr(value))

# Check whether the original CSV actually contains blank IDs
raw_ids = raw_risk_df["recall_number"].str.strip()

actual_blank = raw_ids.isna() | raw_ids.eq("")

print("\nActual blank IDs:", int(actual_blank.sum()))

print("Unique original IDs:", raw_ids.nunique())

print("\nINVESTIGATION COMPLETE")


In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# CELL 4 — MYSQL CONNECTION AND DATABASE INSPECTION
# ============================================================

import mysql.connector
from getpass import getpass
import pandas as pd

print("=" * 70)
print("PHARMAOPT — MYSQL DATABASE INSPECTION")
print("=" * 70)

# 1. Connect securely to MySQL
db_password = getpass("Enter your MySQL password: ")

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=db_password,
    database="pharmaopt"
)

del db_password

cursor = connection.cursor(dictionary=True)

print("\nMySQL connection successful.")
print("Database:", connection.database)

# 2. Identify tables required for model integration
tables = [
    "drug_product",
    "drug_package",
    "shortage_event",
    "recall_event",
    "recall_package"
]

# 3. Inspect table sizes
print("\nDATABASE TABLE COUNTS")
print("-" * 70)

for table in tables:
    cursor.execute(f"SELECT COUNT(*) AS total FROM `{table}`")
    result = cursor.fetchone()
    print(f"{table}: {result['total']:,} records")

# 4. Retrieve actual column names and types
schema_query = """
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    DATA_TYPE,
    IS_NULLABLE,
    COLUMN_KEY
FROM information_schema.COLUMNS
WHERE TABLE_SCHEMA = %s
AND TABLE_NAME IN (
    'drug_product',
    'drug_package',
    'shortage_event',
    'recall_event',
    'recall_package'
)
ORDER BY TABLE_NAME, ORDINAL_POSITION
"""

cursor.execute(schema_query, ("pharmaopt",))

schema_rows = cursor.fetchall()

schema_df = pd.DataFrame(schema_rows)

# 5. Display recall-related schemas
print("\nRECALL EVENT TABLE STRUCTURE")
display(
    schema_df[
        schema_df["TABLE_NAME"] == "recall_event"
    ]
)

print("\nRECALL PACKAGE TABLE STRUCTURE")
display(
    schema_df[
        schema_df["TABLE_NAME"] == "recall_package"
    ]
)

# 6. Identify possible date columns
print("\nAVAILABLE DATE/TIME COLUMNS")

date_columns = schema_df[
    schema_df["COLUMN_NAME"].str.contains(
        "date|time",
        case=False,
        na=False
    )
]

display(date_columns)

print("\nDATABASE INSPECTION COMPLETE")
print("Ready to design historical qualitative risk joins.")


In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# CELL 5 — DATABASE COLUMN IDENTIFICATION
# ============================================================

print("=" * 70)
print("PHARMAOPT — DATABASE STRUCTURE")
print("=" * 70)

tables_to_inspect = [
    "recall_event",
    "recall_package",
    "drug_package",
    "shortage_event"
]

for table in tables_to_inspect:

    print(f"\n{'=' * 60}")
    print(f"TABLE: {table}")
    print("=" * 60)

    columns = schema_df[
        schema_df["TABLE_NAME"] == table
    ][
        ["COLUMN_NAME", "DATA_TYPE", "IS_NULLABLE", "COLUMN_KEY"]
    ]

    print(columns.to_string(index=False))

print("\nSCHEMA REVIEW COMPLETE")


In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# CELL 6 — HISTORICAL RECALL RISK INTEGRATION
# ============================================================

import pandas as pd

print("=" * 70)
print("PHARMAOPT — HISTORICAL RECALL RISK INTEGRATION")
print("=" * 70)

# 1. Retrieve recall-to-package relationships
query = """
SELECT
    re.recall_number,
    re.report_date,
    re.recall_initiation_date,
    rp.source_package_ndc,
    rp.linked_package_ndc
FROM recall_event AS re
INNER JOIN recall_package AS rp
    ON re.recall_number = rp.recall_number
WHERE rp.linked_package_ndc IS NOT NULL
    AND TRIM(rp.linked_package_ndc) <> ''
"""

cursor.execute(query)

recall_rows = cursor.fetchall()

recall_history = pd.DataFrame(recall_rows)

print("\nDatabase recall-package rows:", len(recall_history))

# 2. Standardize identifiers
recall_history["recall_number"] = (
    recall_history["recall_number"]
    .astype("string")
    .str.strip()
)

recall_history["linked_package_ndc"] = (
    recall_history["linked_package_ndc"]
    .astype("string")
    .str.strip()
)

# 3. Convert historical dates
recall_history["report_date"] = pd.to_datetime(
    recall_history["report_date"],
    errors="coerce"
)

recall_history["recall_initiation_date"] = pd.to_datetime(
    recall_history["recall_initiation_date"],
    errors="coerce"
)

# 4. Remove duplicate recall-package relationships
before_dedup = len(recall_history)

recall_history = recall_history.drop_duplicates(
    subset=["recall_number", "linked_package_ndc"]
).copy()

print(
    "Duplicate relationships removed:",
    before_dedup - len(recall_history)
)

# 5. Connect qualitative risk classifications
risk_lookup = risk_df[
    ["recall_number"]
    + risk_categories
    + ["risk_signal_count_v2", "unclassified_risk_v2"]
].copy()

integrated_recalls = recall_history.merge(
    risk_lookup,
    on="recall_number",
    how="left",
    validate="many_to_one",
    indicator=True
)

# 6. Audit matching
matched_mask = integrated_recalls["_merge"].eq("both")

print("\nQUALITATIVE RISK MATCHING")
print("-" * 70)

print(
    "Recall-package relationships:",
    len(integrated_recalls)
)

print(
    "Relationships with qualitative risk signals:",
    int(matched_mask.sum())
)

print(
    "Relationships without qualitative risk signals:",
    int((~matched_mask).sum())
)

# 7. Audit historical dates
print("\nHISTORICAL DATE VALIDATION")
print("-" * 70)

print(
    "Missing report dates:",
    int(integrated_recalls["report_date"].isna().sum())
)

print(
    "Missing initiation dates:",
    int(integrated_recalls["recall_initiation_date"].isna().sum())
)

# 8. Retain matched recalls with usable report dates
usable_mask = (
    matched_mask
    & integrated_recalls["report_date"].notna()
)

qualitative_recall_history = (
    integrated_recalls.loc[usable_mask]
    .drop(columns="_merge")
    .copy()
)

assert not qualitative_recall_history.empty

assert (
    qualitative_recall_history[
        ["recall_number", "linked_package_ndc"]
    ].duplicated().sum() == 0
)

# 9. Summarize integration
print("\nHISTORICAL INTEGRATION SUMMARY")
print("-" * 70)

print(
    "Usable historical relationships:",
    len(qualitative_recall_history)
)

print(
    "Unique linked drug packages:",
    qualitative_recall_history["linked_package_ndc"].nunique()
)

print(
    "Unique recall events:",
    qualitative_recall_history["recall_number"].nunique()
)

# 10. Inspect recall years
print("\nRECALL REPORT YEARS")

print(
    qualitative_recall_history["report_date"]
    .dt.year
    .value_counts()
    .sort_index()
    .to_string()
)

# 11. Preview integrated dataset
print("\nINTEGRATED DATA PREVIEW")

display(
    qualitative_recall_history[
        [
            "recall_number",
            "linked_package_ndc",
            "report_date",
            "risk_signal_count_v2"
        ]
        + risk_categories[:5]
    ].head(10)
)

print("\nCELL 6 COMPLETE")


In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# CELL 7 — TEMPORAL QUALITATIVE RISK FEATURES
# ============================================================

import pandas as pd

print("=" * 70)
print("PHARMAOPT — TEMPORAL RISK FEATURE ENGINEERING")
print("=" * 70)

# 1. Prepare historical recall data
required_cols = [
    "recall_number",
    "linked_package_ndc",
    "report_date",
    "unclassified_risk_v2"
] + risk_categories

history = qualitative_recall_history[
    required_cols
].copy()

history["report_date"] = pd.to_datetime(
    history["report_date"],
    errors="raise"
)

assert history["report_date"].notna().all()

assert not history[
    ["recall_number", "linked_package_ndc"]
].duplicated().any()

# 2. Validate qualitative category values
for col in risk_categories + ["unclassified_risk_v2"]:
    history[col] = pd.to_numeric(
        history[col],
        errors="raise"
    ).astype(int)

    assert history[col].isin([0, 1]).all()

# 3. Define historical prediction years
prediction_years = [2022, 2023, 2024, 2025]

feature_frames = []

# 4. Build features separately for each year
for year in prediction_years:

    cutoff = pd.Timestamp(year=year, month=1, day=1)

    start_3y = cutoff - pd.DateOffset(years=3)

    start_1y = cutoff - pd.DateOffset(years=1)

    # Only use recalls reported BEFORE prediction year
    prior = history.loc[
        (history["report_date"] >= start_3y)
        & (history["report_date"] < cutoff)
    ].copy()

    assert (prior["report_date"] < cutoff).all()

    # Flag recalls within the last year
    prior["recall_in_last_year"] = (
        prior["report_date"] >= start_1y
    ).astype(int)

    # Aggregate recall information by drug package
    aggregation = {
        "prior_recall_count_3y": (
            "recall_number", "nunique"
        ),
        "prior_recall_count_1y": (
            "recall_in_last_year", "sum"
        ),
        "prior_unclassified_count_3y": (
            "unclassified_risk_v2", "sum"
        )
    }

    # Add historical counts for all 13 risk categories
    for category in risk_categories:
        aggregation[f"prior_{category}_count_3y"] = (
            category, "sum"
        )

    yearly_features = (
        prior.groupby(
            "linked_package_ndc",
            as_index=False
        )
        .agg(**aggregation)
    )

    yearly_features["year"] = year

    feature_frames.append(yearly_features)

    print(
        f"{year}: "
        f"{len(prior):,} historical relationships, "
        f"{len(yearly_features):,} packages"
    )

# 5. Combine all prediction years
qualitative_features = pd.concat(
    feature_frames,
    ignore_index=True
)

# 6. Validate package-year uniqueness
assert not qualitative_features[
    ["linked_package_ndc", "year"]
].duplicated().any()

assert (
    qualitative_features["prior_recall_count_1y"]
    <= qualitative_features["prior_recall_count_3y"]
).all()

# 7. Save historical features
feature_file = (
    OUTPUT_DIR
    / "recall_risk_features_package_year_2022_2025_v05.csv"
)

qualitative_features.to_csv(
    feature_file,
    index=False
)

# 8. Display results
print("\n" + "=" * 70)
print("TEMPORAL FEATURE ENGINEERING COMPLETE")
print("=" * 70)

print("Package-year records:", len(qualitative_features))
print("Risk categories:", len(risk_categories))
print("Total feature columns:", len(qualitative_features.columns))
print("Duplicate package-years:",
      int(qualitative_features[
          ["linked_package_ndc", "year"]
      ].duplicated().sum()))

print("File saved:", feature_file)
print("File exists:", feature_file.exists())

print("\nPACKAGE-YEAR COUNTS")
print(
    qualitative_features["year"]
    .value_counts()
    .sort_index()
    .to_string()
)

display(qualitative_features.head(10))


In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# CELL 8 — LOCATE EXISTING MODEL DATA
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 70)
print("PHARMAOPT — MODEL DATA DISCOVERY")
print("=" * 70)

# 1. Confirm project location
print("\nPROJECT ROOT")
print(PROJECT_ROOT)

# 2. Inspect output files
print("\nEXISTING OUTPUT FILES")
print("-" * 70)

output_files = sorted(
    [p for p in OUTPUT_DIR.rglob("*") if p.is_file()]
)

print("Total output files:", len(output_files))

for file in output_files:
    size_mb = file.stat().st_size / (1024 ** 2)

    print(
        f"{file.relative_to(PROJECT_ROOT)} "
        f"({size_mb:.2f} MB)"
    )

# 3. Find files that may contain model data
keywords = [
    "panel",
    "model",
    "prediction",
    "shortage",
    "training",
    "feature",
    "cms",
    "v04",
    "v0.4"
]

candidate_files = [
    p for p in output_files
    if any(word in p.name.lower() for word in keywords)
]

print("\nPOTENTIAL MODEL DATA FILES")
print("-" * 70)

for file in candidate_files:
    print(file.name)

# 4. Inspect CSV column names without loading full datasets
print("\nCANDIDATE CSV STRUCTURES")
print("-" * 70)

for file in candidate_files:

    if file.suffix.lower() != ".csv":
        continue

    try:
        columns = pd.read_csv(
            file,
            nrows=0
        ).columns.tolist()

        print("\nFile:", file.name)
        print("Columns:", columns[:35])

    except Exception as error:
        print("Could not inspect:", file.name)
        print(type(error).__name__, str(error))

# 5. Locate Notebook 06
print("\nNOTEBOOK 06 LOCATION")
print("-" * 70)

notebook_matches = list(
    PROJECT_ROOT.glob("06_*")
) + list(
    (PROJECT_ROOT / "notebooks").glob("06_*")
)

for notebook in notebook_matches:
    print(notebook)

if not notebook_matches:
    print("Notebook 06 not found in expected folders.")

print("\nCELL 8 COMPLETE")


In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# CELL 9 — DISCOVER EXISTING MODEL STRUCTURE
# ============================================================

import json
import re
from pathlib import Path

print("=" * 70)
print("PHARMAOPT — EXISTING MODEL STRUCTURE DISCOVERY")
print("=" * 70)

# 1. Locate Notebook 06
notebook_files = sorted(
    PROJECT_ROOT.glob("06_pharmaopt_analytics*")
)

if not notebook_files:
    notebook_files = sorted(
        (PROJECT_ROOT / "notebooks").glob(
            "06_pharmaopt_analytics*"
        )
    )

if not notebook_files:
    raise FileNotFoundError(
        "Notebook 06 could not be located."
    )

notebook_path = notebook_files[0]

print("\nNotebook found:", notebook_path.name)

# 2. Read notebook without executing it
with open(notebook_path, "r", encoding="utf-8") as file:
    notebook_data = json.load(file)

cells = notebook_data.get("cells", [])

print("Total notebook cells:", len(cells))

# 3. Identify model-building code
search_terms = [
    "panel",
    "feature_cols",
    "feature_columns",
    "model_df",
    "training_df",
    "train_mask",
    "test_mask",
    "x_train",
    "x_test",
    "predict_proba",
    "v0.4",
    "v04",
    "cms",
    "to_csv"
]

# Exclude lines that might contain credentials
sensitive_terms = [
    "password",
    "secret",
    "api_key",
    "access_token",
    "getpass"
]

print("\nMODEL-RELATED CODE REFERENCES")
print("-" * 70)

matches_found = 0
max_matches = 65

for cell_number, cell in enumerate(cells, start=1):

    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])

    if isinstance(source, list):
        source = "".join(source)

    for line_number, line in enumerate(
        source.splitlines(), start=1
    ):

        line_lower = line.lower()

        if any(
            term in line_lower
            for term in search_terms
        ):

            if any(
                term in line_lower
                for term in sensitive_terms
            ):
                continue

            print(
                f"Cell {cell_number}, "
                f"line {line_number}: "
                f"{line.strip()[:160]}"
            )

            matches_found += 1

            if matches_found >= max_matches:
                break

    if matches_found >= max_matches:
        break

print("\nTotal references displayed:", matches_found)

if matches_found == max_matches:
    print("Output limited to first 65 matches.")

print("\nCELL 9 COMPLETE")
print("No notebooks or database tables were modified.")


In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# CELL 10 — INSPECT EXISTING MODEL CSV FILES
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 70)
print("PHARMAOPT — EXISTING MODEL DATASETS")
print("=" * 70)

# Find PharmaOpt CSV files in the project folder
model_files = sorted(
    PROJECT_ROOT.glob("pharmaopt*.csv")
)

print("\nFiles found:", len(model_files))

if not model_files:
    print("No PharmaOpt CSV files found in project root.")

for file in model_files:

    print("\n" + "=" * 70)
    print("FILE:", file.name)
    print("=" * 70)

    print(
        "Size:",
        round(file.stat().st_size / (1024 ** 2), 2),
        "MB"
    )

    # Read only 3 rows, preserving identifiers as text
    preview = pd.read_csv(
        file,
        nrows=3,
        dtype=str
    )

    print("\nCOLUMNS:")
    for col in preview.columns:
        print(" -", col)

    print("\nIMPORTANT COLUMNS:")
    important = [
        col for col in preview.columns
        if any(
            word in col.lower()
            for word in [
                "ndc",
                "year",
                "target",
                "shortage",
                "cms",
                "risk"
            ]
        )
    ]

    print(important)

    print("\nPREVIEW:")
    display(preview[important].head(3) if important
            else preview.head(3))

print("\nCELL 10 COMPLETE")
print("No existing files were changed.")


In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# CELL 11 — MODEL SOURCE AUDIT
# ============================================================

import pandas as pd
import json
from pathlib import Path

print("=" * 70)
print("PHARMAOPT — MODEL SOURCE AUDIT")
print("=" * 70)

# 1. Inspect existing PharmaOpt CSV files
model_files = sorted(
    PROJECT_ROOT.glob("pharmaopt*.csv")
)

print("\nEXISTING MODEL DATA FILES")
print("-" * 70)

for file in model_files:

    columns = pd.read_csv(
        file,
        nrows=0
    ).columns.tolist()

    print("\nFILE:", file.name)
    print("COLUMN COUNT:", len(columns))

    print("HAS PACKAGE NDC:", "package_ndc" in columns)
    print("HAS PREDICTION YEAR:", "prediction_year" in columns)
    print("HAS SHORTAGE TARGET:", "shortage_target" in columns)
    print("HAS MODEL RISK SCORE:", "risk_score" in columns)

    print("\nALL COLUMNS:")
    for column in columns:
        print(" -", column)

# 2. Read Notebook 06 source without executing it
notebooks = sorted(
    PROJECT_ROOT.glob("06_pharmaopt_analytics*")
)

if not notebooks:
    raise FileNotFoundError("Notebook 06 not found.")

with open(notebooks[0], "r", encoding="utf-8") as f:
    notebook = json.load(f)

# 3. Find the code that built the v0.4 dataset
search_terms = [
    "pharmaopt_v04_df",
    "pharmaopt_v03_df",
    "features_v04",
    "train_v04",
    "test_v04",
    "to_csv",
    "prediction_year"
]

print("\n" + "=" * 70)
print("NOTEBOOK 06 — v0.4 MODEL CONSTRUCTION")
print("=" * 70)

for cell_number, cell in enumerate(
    notebook["cells"], start=1
):

    if cell_number < 32:
        continue

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    matches = [
        line.strip()
        for line in source.splitlines()
        if any(
            term in line.lower()
            for term in search_terms
        )
        and not any(
            sensitive in line.lower()
            for sensitive in [
                "password",
                "secret",
                "api_key",
                "access_token"
            ]
        )
    ]

    if matches:
        print(f"\nNOTEBOOK CELL {cell_number}")

        for line in matches[:20]:
            print(" ", line[:180])

print("\nCELL 11 COMPLETE")
print("No files or database tables were modified.")


In [ ]:

# ============================================================
# PHARMAOPT — NOTEBOOK 10
# CELL 12 — MODEL REBUILD DEPENDENCY AUDIT
# ============================================================

import json
import re

print("=" * 70)
print("PHARMAOPT — MODEL REBUILD DEPENDENCY AUDIT")
print("=" * 70)

notebook_path = sorted(
    PROJECT_ROOT.glob("06_pharmaopt_analytics*")
)[0]

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Cells responsible for the modeling pipeline
target_cells = [18, 25, 26, 29, 32, 33, 34]

patterns = [
    r"\bpd\.read_",
    r"\bread_sql\b",
    r"\bcursor\.execute\b",
    r"\bconnect\(",
    r"\bSELECT\b",
    r"\bFROM\b",
    r"\bJOIN\b",
    r"\bpharmaopt_\w+\s*=",
    r"\bfeatures_\w+\s*=",
    r"\btrain_\w+\s*=",
    r"\btest_\w+\s*=",
    r"\bPROJECT_DIR\b",
    r"\bmerge\(",
    r"\bconcat\(",
    r"\bgroupby\(",
    r"\bto_csv\(",
    r"\bto_sql\(",
    r"\bINSERT\b",
    r"\bUPDATE\b",
    r"\bDELETE\b",
    r"\bCREATE\b",
    r"\bDROP\b"
]

combined_pattern = re.compile(
    "|".join(patterns),
    flags=re.IGNORECASE
)

sensitive_pattern = re.compile(
    r"password|secret|api[_-]?key|access[_-]?token",
    flags=re.IGNORECASE
)

for cell_number in target_cells:

    cell = notebook["cells"][cell_number - 1]

    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    source = "".join(source) if isinstance(source, list) else source

    lines = source.splitlines()

    print("\n" + "=" * 70)
    print(f"NOTEBOOK 06 — CELL {cell_number}")
    print("=" * 70)

    # Show opening context, omitting potential credentials
    print("\nOPENING CODE:")

    for line_number, line in enumerate(lines[:12], start=1):
        if not sensitive_pattern.search(line):
            print(f"{line_number:>3}: {line[:140]}")

    print("\nDATA / MODEL DEPENDENCIES:")

    matches = 0

    for line_number, line in enumerate(lines, start=1):

        if sensitive_pattern.search(line):
            continue

        if combined_pattern.search(line):

            print(f"{line_number:>3}: {line.strip()[:140]}")
            matches += 1

            if matches >= 35:
                print("... additional matches omitted")
                break

print("\nCELL 12 COMPLETE")
print("Inspection only — no code was executed from Notebook 06.")
print("No database tables or files were modified.")
